# [Step 5.5] 커널 기반 이상 탐지 + Conformal Prediction

> Step 5(UQ) 와 Step 6(배포) 사이. 물리로 잡을 수 있으면 exact, 못 잡으면 distribution-free.

## A1. 잔차(residual)의 통계적 성질

칼만 필터는 매 스텝 혁신(innovation) 을 만듭니다.

$$ y_k = z_k - H\hat{x}_k^- , \qquad S_k = H P_k^- H^T + R $$

H0(모델이 맞고 정상 비행)에서 $y_k$ 는 영평균 백색 가우시안이고 공분산은 $S_k$ 입니다.
따라서 정규화 잔차 $z_k = y_k/\sqrt{S_k}$ 는 표준정규 $\mathcal{N}(0,1)$ 을 따릅니다.

이 성질이 두 가지를 가능하게 합니다.

| 채널 | 통계량 | 보장 |
|:--|:--|:--|
| L1 | $\chi^2 = \sum_k z_k^2$ (창 내 합) | 이론적 exact — $\chi^2(W)$ 분포 |
| L2 | 다변량 창 피처 → one-class 모델 | distribution-free — CP 로 유한표본 보장 |

⚠️ L1 의 exact 보장은 모델이 완전할 때만 성립합니다. 명목 파라미터 모델에는 모델 오차가
있어 실제 FPR 이 이론값을 초과합니다 — 아래 B9 에서 실측합니다.

## A2. χ² 검정이 잡는 것 vs 못 잡는 것

### 두 종류의 필터, 두 종류의 이상

| 필터 | 상태 예측 근거 | 무엇에 반응하는가 |
|:--|:--|:--|
| 기구학 추적기 (Step 2) | 측정된 가속도 $u$ 를 그대로 적분 | 센서 결함만 |
| 명목 물리모델 (이 노트북) | 명목 질량·Cd·발사각으로 동역학 적분 | 비행체·파라미터 이상 |

Step 2 의 필터는 측정 가속도를 입력으로 쓰므로 어떤 궤적이든 "그게 정답"으로 따라갑니다.
파라미터가 달라져도 잔차에는 센서 잡음만 남습니다.

> 실측 (정상 200편 vs 파라미터 이탈 OOD 50편)
> · Step 2 기구학 필터 잔차 비 = 1.00x → 탐지 불가
> · 명목 물리모델 잔차 비 = 2.05x → 탐지 가능
>
> 즉 "혁신이 이상에서 커진다"는 성질은 모델 기반 잔차에서만 성립합니다.
> 이것이 이 노트북이 별도 물리모델 필터(`scripts/model_filter.py`)를 쓰는 이유입니다.

### Mehra (1971) 의 고전적 결과

혁신의 통계적 성질을 이용한 고장 검출은 1970년대부터 정립된 기법입니다.
핵심은 H0 에서 혁신이 백색이라는 성질이 깨지는 순간을 탐지하는 것.
한계도 분명합니다 — 비가우시안 잔차, 센서 포화, 모델 오차는 χ² 로 잡히지 않습니다.
그 회색지대를 L2(커널 + CP)가 담당합니다.

## A3. ML2 복습 — one-class 4종

정상 데이터만 있고 고장 데이터가 없습니다. 게다가 실측은 소표본입니다.
이 조건이 one-class 방법의 홈그라운드입니다.

| 방법 | 원리 | 비선형 | ONNX 변환 | 온보드 |
|:--|:--|:--:|:--:|:--:|
| OC-SVM (Schölkopf 2001) | 원점에서 최대 마진 초평면 | RBF 커널 | ✅ | ✅ 주축 |
| SVDD (Tax & Duin 2004) | 정상 데이터 최소 구 | RBF 커널 | ✅ | ✅ (OC-SVM 과 근사 동치) |
| KDE | 밀도 추정 후 저밀도 = 이상 | - | ❌ `MissingShapeCalculator` | ❌ 기준선만 |
| kPCA | 재구성 오차 | RBF 커널 | ✅ | △ 성능·비용 열위 |

합성 데이터 4시나리오 ROC-AUC 실측 (선행 검증, 2026-08-18):

| 시나리오 | OC-SVM | KDE | kPCA |
|:--|:--:|:--:|:--:|
| 2D 멀티모달 | 0.893 | 0.926 | 0.902 |
| 20D 평균이동 | 0.990 | 1.000 | 0.985 |
| n=60 소표본 | 0.934 | 0.948 | 0.937 |
| 2D 균등정상 | 0.982 | 0.989 | 0.976 |

AUC 는 KDE 가 최고지만 탑재 비용이 결정적입니다.
추론 1000샘플(10D, n=2000): OC-SVM 12.8 ms vs KDE 240.4 ms vs kPCA 158.9 ms.
OC-SVM 은 서포트 벡터만 저장(sparse)하고 ONNX 변환이 되므로 주축은 OC-SVM(RBF) + CP,
KDE 는 비교 기준선으로만 씁니다.

## A4. Conformal Prediction — 분포 가정 없는 보장

임계값을 "경험적으로" 정하면 오탐률을 보장할 수 없습니다.
CP 는 교환가능성(exchangeability) 만 가정하고 유한표본 보장을 줍니다.

### Split conformal (Vovk 2005; Angelopoulos & Bates 2021)

1. 비적합도 점수(nonconformity score) $s(x)$ 를 정한다 — 클수록 이상
2. calibration 셋 $\{s_1,\dots,s_n\}$ 의 분위수를 취한다

$$ \hat{q} = \text{Quantile}_{\lceil (n+1)(1-\alpha)\rceil / n}\big(\{s_i\}\big) $$

3. 새 점이 $s(x) > \hat{q}$ 이면 이상으로 판정

보장: 교환가능한 데이터에서 $\mathbb{P}(s(x_{new}) > \hat{q}) \le \alpha$ (marginal, 유한표본).

### 이 노트북의 스코어 정의

OC-SVM 의 `decision_function` 은 양수가 정상, 음수가 이상입니다. 부호를 뒤집습니다.

$$ s(x) = -\text{decision\_function}(x) $$

> ⚠️ ONNX score offset 함정 (sklearn-onnx #912). sklearn 의 `score_samples(X)` 는
> `decision_function(X) + offset_` (shifted)이고 ONNX 출력은 `decision_function(X)` (unshifted)입니다.
> RBF 실측 차이 1.13 vs 0.24 (offset 0.89). 상수 시프트라 순위/AUC 는 무관하지만
> 절대 임계값이 달라집니다 → 스코어 정의를 `decision_function` 기준으로 통일해야
> 배포 시 임계값이 보존됩니다. 아래 B8 이 그렇게 구현합니다.

## A5. 계층적 인증 — 물리 exact + ML distribution-free

```
L0  물리 경계        센서값이 물리적 하드 바운드 밖        → 즉시 abort (결정적)
L1  χ² 검정 (잔차)    모델이 잡는 오류                    → 즉시 abort (결정적, exact)
L2  OC-SVM + CP      비가우시안·모델오차 잔차             → N-of-M: 최근 5창 중 3창 이상
                                                          → abort (ML, distribution-free)
```

L2 는 N-of-M 으로 지연 필터(lag filter)로 동작합니다. 창 하나의 오탐이 곧바로 abort 를
일으키지 않습니다. 해제는 5연속 정상을 요구합니다(히스테리시스).

이 결정 로직은 이미 `safe_logic.py` 에 구현되어 있습니다 —
`SafeLogic(n=3, m=5, hysteresis_clear=5)`, `l1_chi2_flag()`, `l2_cp_flag()`.

> 근거 (결정 D): 오탐 = 차량 파괴이므로 결정적 경로(L0/L1)가 최종 결정을 갖고,
> ML(L2)은 보조합니다. N-of-M 규칙 자체를 하나의 conformal 객체로 calibration 하는 것이
> 이 스텝의 연구 포인트입니다 (비행 단위 FPR).

In [1]:
# ============================================================
# [B1] 잔차·메타데이터 로드
# ============================================================
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# 한글 폰트 (OS별 자동 선택 — step1.5 / step2 와 동일 관례)
from matplotlib import font_manager as _fm
_avail = {f.name for f in _fm.fontManager.ttflist}
for _n in ('Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR', 'Noto Sans KR'):
    if _n in _avail:
        plt.rcParams['font.family'] = _n
        break
plt.rcParams['axes.unicode_minus'] = False

sys.path.insert(0, os.path.abspath('.'))

RDP = './data/simulated/residuals.npz'
if not os.path.exists(RDP):
    raise FileNotFoundError(
        f"{RDP} 가 없습니다. 먼저 실행하세요:  python scripts/model_filter.py")

_d = np.load(RDP, allow_pickle=False)
flat = pd.DataFrame(_d['flat'], columns=[str(c) for c in _d['cols']])
fl = pd.DataFrame({'fid': _d['fl_ids'].astype(int),
                   'ood': _d['fl_ood'].astype(int),
                   'type': [str(t) for t in _d['fl_type']]})

# 잔차의 두 채널: 명목 물리모델 잔차(L1/L2 입력)와 Step 2 기구학 혁신(참고)
_inn = './data/simulated/innovations.npz'
HAS_KINEMATIC = os.path.exists(_inn)

print(f"잔차 데이터: {len(flat)}행 / {flat['flight_id'].nunique()}편")
print(f"  정상 {(fl.ood == 0).sum()}편 / OOD {(fl.ood == 1).sum()}편")
print(f"  OOD 계열: {fl.loc[fl.ood == 1, 'type'].value_counts().to_dict()}")
print(f"  컬럼: {list(flat.columns)}")
print(f"  Step 2 기구학 혁신 파일: {'있음' if HAS_KINEMATIC else '없음'}")
print()
print(flat.head(3).to_string(index=False))

잔차 데이터: 82097행 / 250편
  정상 200편 / OOD 50편
  OOD 계열: {'drag': 15, 'mass': 15, 'wind': 11, 'loft': 9}
  컬럼: ['flight_id', 'time', 'y', 'S', 'z', 'is_powered']
  Step 2 기구학 혁신 파일: 있음

 flight_id  time        y        S        z  is_powered
       0.0  0.00 0.000000 8.000000 0.000000         1.0
       0.0  0.02 1.000000 6.002600 0.408160         1.0
       0.0  0.04 0.168073 5.339221 0.072737         1.0


In [2]:
# ============================================================
# [B2] 창(window) 피처 구성
# ============================================================
# 0.02 s x 25 = 0.5 초 창. 창 하나가 이상 탐지의 최소 단위가 된다.
W, STRIDE = 25, 10
FEAT = ['z_mean', 'z_std', 'z_max', 'z_min', 'z_rms',
        'y_mean_abs', 'y_max_abs', 'y_std']

def make_windows(flat, W=W, S=STRIDE):
    # 비행별로 창을 잘라 피처 · chi2 통계량 · 위상 · id 를 만든다
    X, CHI, PH, ID = [], [], [], []
    for fid, g in flat.groupby('flight_id'):
        g = g.sort_values('time')
        zz = g['z'].to_numpy(); yy = g['y'].to_numpy(); pw = g['is_powered'].to_numpy()
        n = len(g)
        for a in range(0, n - W + 1, S):
            zw, yw = zz[a:a+W], yy[a:a+W]
            X.append([zw.mean(), zw.std(), np.abs(zw).max(), np.abs(zw).min(),
                      np.sqrt((zw ** 2).mean()),
                      np.abs(yw).mean(), np.abs(yw).max(), yw.std()])
            CHI.append(float((zw ** 2).sum()))     # L1 χ² 통계량
            PH.append(float(pw[a:a+W].mean()))     # 위상: 1=연소중, 0=관성
            ID.append(int(fid))
    return np.array(X), np.array(CHI), np.array(PH), np.array(ID)

Xw, CHIw, PHw, IDw = make_windows(flat)
meta = fl.set_index('fid').loc[IDw]
IS_OOD = meta['ood'].to_numpy()
OOD_TYPE = meta['type'].to_numpy()

print(f"창 {len(Xw)}개 / 피처 {Xw.shape[1]}개")
print(f"  정상 창 {int((IS_OOD==0).sum())} / OOD 창 {int((IS_OOD==1).sum())}")
print(f"  χ² 통계량(연소중) 중앙 {np.median(CHIw[PHw>0.5]):.1f} / (관성) 중앙 {np.median(CHIw[PHw<0.5]):.1f}")
print(f"  참고: H0 이면 χ²({W}) 기대값 = {W}")

창 7720개 / 피처 8개
  정상 창 6351 / OOD 창 1369
  χ² 통계량(연소중) 중앙 17.9 / (관성) 중앙 31.2
  참고: H0 이면 χ²(25) 기대값 = 25


In [3]:
# ============================================================
# [B3] 비행 단위 분할 ⚠️ 샘플 단위가 아니다
# ============================================================
# 한 비행 안의 창들은 강하게 상관된다. 샘플 단위로 섞으면
# 같은 비행의 창이 train/test 에 동시에 들어가 누수가 생기고
# CP 의 교환가능성 가정이 깨진다. 반드시 **비행 단위**로 나눈다.
SEED = 5505
ALPHA = 0.10                      # 목표 오탐률 10%
rng = np.random.default_rng(SEED)

norm_ids = np.sort(fl.loc[fl.ood == 0, 'fid'].to_numpy())
ood_ids = np.sort(fl.loc[fl.ood == 1, 'fid'].to_numpy())
perm = rng.permutation(norm_ids)
TRAIN_IDS, CAL_IDS, TEST_IDS = perm[:100], perm[100:150], perm[150:]

M_TRAIN = np.isin(IDw, TRAIN_IDS)
M_CAL = np.isin(IDw, CAL_IDS)
M_TEST = np.isin(IDw, TEST_IDS)
M_OOD = np.isin(IDw, ood_ids)

print(f"비행 단위 분할 (총 정상 {len(norm_ids)}편)")
print(f"  train      {len(TRAIN_IDS):3d}편 → 창 {M_TRAIN.sum():5d}")
print(f"  calibration{len(CAL_IDS):3d}편 → 창 {M_CAL.sum():5d}")
print(f"  test(정상) {len(TEST_IDS):3d}편 → 창 {M_TEST.sum():5d}")
print(f"  test(OOD)  {len(ood_ids):3d}편 → 창 {M_OOD.sum():5d}")
print(f"\n  군집 여부 확인: 창 중복 비행 = {pd.Series(IDw[M_TRAIN]).isin(TEST_IDS).sum()} (0 이어야 정상)")

# 스케일러는 train 창으로만 fit (누수 방지)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(Xw[M_TRAIN])
def S(mask): return scaler.transform(Xw[mask])

비행 단위 분할 (총 정상 200편)
  train      100편 → 창  3162
  calibration 50편 → 창  1579
  test(정상)  50편 → 창  1610
  test(OOD)   50편 → 창  1369

  군집 여부 확인: 창 중복 비행 = 0 (0 이어야 정상)


In [4]:
# ============================================================
# [B4] OC-SVM (RBF) — 주축 모델
# ============================================================
from sklearn.svm import OneClassSVM

# nu: 정상 밖으로 밀려날 허용 비율의 상한. 느슨하게 두고 최종 임계는 CP 가 결정한다.
NU = 0.10
ocsvm = OneClassSVM(kernel='rbf', nu=NU, gamma='scale').fit(S(M_TRAIN))

d_tr = ocsvm.decision_function(S(M_TRAIN))
print(f"OC-SVM 학습 완료  (nu={NU}, RBF)")
print(f"  서포트 벡터 {ocsvm.support_vectors_.shape[0]} / 학습 {M_TRAIN.sum()}  "
      f"({ocsvm.support_vectors_.shape[0]/M_TRAIN.sum()*100:.1f}% — sparse 저장 가능)")
print(f"  train decision_function: 평균 {d_tr.mean():+.4f} / 표준편차 {d_tr.std():.4f}")
print(f"  → 양수 = 정상, 음수 = 이상 (sklearn 관례)")

# 비적합도 스코어: 클수록 이상
score_svm = lambda Xs: -ocsvm.decision_function(Xs)

OC-SVM 학습 완료  (nu=0.1, RBF)
  서포트 벡터 324 / 학습 3162  (10.2% — sparse 저장 가능)
  train decision_function: 평균 +2.2167 / 표준편차 3.4497
  → 양수 = 정상, 음수 = 이상 (sklearn 관례)


## B5. SVDD 와의 관계 (교육용)

SVDD(Support Vector Data Description, Tax & Duin 2004)는 정상 데이터를 감싸는
최소 반지름 구를 찾습니다. OC-SVM 은 원점을 지나는 최대 마진 초평면을 찾습니다.

RBF 커널에서는 두 문제가 거의 동치입니다 — 커널 공간에서 원점을 지나는 초평면은
입력 공간에서 구(비선형 경계)에 대응하기 때문입니다.

| | OC-SVM | SVDD |
|:--|:--|:--|
| 경계 | 원점 기준 초평면 | 중심 기준 구 |
| 파라미터 | $\nu$, $\gamma$ | $C$, $R$ |
| sklearn | `OneClassSVM` | 전용 클래스 없음 (`SVDD` 미제공) |
| 실무 | 이걸 쓴다 | 개념 이해용 |

아래 셀에서 두 관점의 대응을 수치로 확인합니다.

In [5]:
# ============================================================
# [B5] SVDD 관점 확인 — 결정함수 분포와 마진
# ============================================================
d_cal = ocsvm.decision_function(S(M_CAL))
d_tst = ocsvm.decision_function(S(M_TEST))
d_ood = ocsvm.decision_function(S(M_OOD))

print("decision_function 분포 (양수=정상 쪽)")
for tag, d in (('train', d_tr), ('cal', d_cal), ('test(정상)', d_tst), ('test(OOD)', d_ood)):
    print(f"  {tag:11s} 중앙 {np.median(d):+7.4f}  5~95분위 {np.percentile(d,5):+7.4f} ~ {np.percentile(d,95):+7.4f}")

# '구의 반지름' 대응: 결정함수 0 이 경계
r_eq = np.sqrt(np.mean(np.sum(scaler.transform(Xw[M_TRAIN]) ** 2, axis=1)))
print(f"\n  대응하는 RBF 구 반지름(근사) r = {r_eq:.4f}")
print(f"  경계(decision_function=0) 기준 정상 오탐률: {np.mean(d_tst < 0)*100:.1f}%")
print(f"  경계 기준 OOD 탐지율            : {np.mean(d_ood < 0)*100:.1f}%")
print("  → nu 만으로는 FPR 을 α 로 고정할 수 없다. 그 역할이 아래 CP 다.")

decision_function 분포 (양수=정상 쪽)
  train       중앙 +2.9818  5~95분위 -2.5256 ~ +4.9256
  cal         중앙 +2.9680  5~95분위 -2.3924 ~ +4.8458
  test(정상)    중앙 +2.7916  5~95분위 -3.0412 ~ +4.6583
  test(OOD)   중앙 +2.0149  5~95분위 -36.5950 ~ +4.7291

  대응하는 RBF 구 반지름(근사) r = 2.8284
  경계(decision_function=0) 기준 정상 오탐률: 13.0%
  경계 기준 OOD 탐지율            : 33.6%
  → nu 만으로는 FPR 을 α 로 고정할 수 없다. 그 역할이 아래 CP 다.


In [6]:
# ============================================================
# [B6] KDE — 비교 기준선 (온보드 탈락)
# ============================================================
from sklearn.neighbors import KernelDensity
from sklearn.model_selection import GridSearchCV

# bandwidth 를 calibration 이 아니라 train 내부 CV 로 고른다 (누수 방지)
gs = GridSearchCV(KernelDensity(kernel='gaussian'),
                  {'bandwidth': [0.3, 0.5, 0.8, 1.2, 2.0]}, cv=3).fit(S(M_TRAIN))
kde = gs.best_estimator_
print(f"KDE bandwidth = {gs.best_params_['bandwidth']}")

# 저밀도 = 이상 → 부호 반전
score_kde = lambda Xs: -kde.score_samples(Xs)
print(f"  train 스코어(=-log밀도) 중앙 {np.median(score_kde(S(M_TRAIN))):+.4f}")
print("  ⚠️ KDE 는 ONNX 변환 미지원(MissingShapeCalculator) → 온보드 후보에서 제외")

KDE bandwidth = 0.3


  train 스코어(=-log밀도) 중앙 +3.0626
  ⚠️ KDE 는 ONNX 변환 미지원(MissingShapeCalculator) → 온보드 후보에서 제외


In [7]:
# ============================================================
# [B7] kPCA — 재구성 오차
# ============================================================
from sklearn.decomposition import KernelPCA

kpca = KernelPCA(n_components=10, kernel='rbf', fit_inverse_transform=True,
                 gamma=None, random_state=SEED).fit(S(M_TRAIN))

def score_kpca(Xs):
    return np.linalg.norm(Xs - kpca.inverse_transform(kpca.transform(Xs)), axis=1)

print(f"kPCA 재구성 오차 (train) 중앙 {np.median(score_kpca(S(M_TRAIN))):.4f}")
print("  → ONNX 변환은 가능하나 성능은 OC-SVM 과 유사하고 추론 비용은 KDE 급 → 비교용")

kPCA 재구성 오차 (train) 중앙 0.3383
  → ONNX 변환은 가능하나 성능은 OC-SVM 과 유사하고 추론 비용은 KDE 급 → 비교용


In [8]:
# ============================================================
# [B8] CP calibration — 임계값을 분포 가정 없이 정한다
# ============================================================
def cp_threshold(scores, alpha):
    # split conformal: ceil((n+1)(1-alpha)) 번째 순서통계량
    s = np.sort(np.asarray(scores))
    n = len(s)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    return float(s[min(k, n) - 1])

# 스코어는 반드시 decision_function 기준 (ONNX offset 함정 회피)
q_svm = cp_threshold(score_svm(S(M_CAL)), ALPHA)
q_kde = cp_threshold(score_kde(S(M_CAL)), ALPHA)
q_kpca = cp_threshold(score_kpca(S(M_CAL)), ALPHA)
q_chi = float(__import__('scipy.stats', fromlist=['chi2']).chi2.ppf(1 - ALPHA, W))

print(f"CP 임계값 (α={ALPHA}, calibration 창 {int(M_CAL.sum())}개)")
print(f"  OC-SVM+CP  q = {q_svm:+8.4f}")
print(f"  KDE+CP     q = {q_kde:+8.4f}")
print(f"  kPCA+CP    q = {q_kpca:+8.4f}")
print(f"  χ² 이론    q = {q_chi:+8.4f}   (χ²({W}) 의 {100*(1-ALPHA):.0f}% 분위)")
print()
print("  ※ χ² 임계는 이론값이라 calibration 이 필요 없다. 대신 모델이 완전할 때만 유효하다.")

CP 임계값 (α=0.1, calibration 창 1579개)
  OC-SVM+CP  q =  +0.1251
  KDE+CP     q =  +5.9157
  kPCA+CP    q =  +0.9377
  χ² 이론    q = +34.3816   (χ²(25) 의 90% 분위)

  ※ χ² 임계는 이론값이라 calibration 이 필요 없다. 대신 모델이 완전할 때만 유효하다.


## B8b. L1 진단 — 이론 χ² 임계가 왜 성립하지 않는가

### χ² 이론 임계의 전제

$\chi^2 = \sum_k z_k^2 \sim \chi^2(W)$ 는 $z_k$ 가 백색(white) 일 때만 성립합니다.
잔차가 서로 상관되어 있으면 합의 분산이 $2W$ 보다 커지고, 분포는 꼬리가 두꺼워집니다
→ 이론 임계가 과도하게 발화합니다.

### 실측 진단

시도한 오진: "모델 오차가 $S$ 를 과소평가한다"고 보고 $\hat\rho = \mathbb{E}[z_{std}^2]$ 로
$S$ 를 스케일 보정했습니다. $\hat\rho = 0.55$ 가 나와 나눴더니 FPR 이 34% → 56% 로 악화.
창 내 표준편차는 저주파 드리프트를 과소평가하므로 분산 추정치로 부적합했습니다.

진짜 원인: 잔차의 자기상관(백색성 위반) 입니다. 명목 모델의 오차는 매끄러운
모델 불일치라 창 안에서 서로 상관됩니다. 아래 셀이 lag-1 자기상관을 측정합니다.

### 그래서 L1 을 어떻게 쓰는가

백색성을 회복하려면 잔차 백색화(whitening) 가 필요합니다(AR 모델 등). 그것은
범위 밖이므로, L1 의 임계값을 calibration 창에서 경험적으로 정합니다.
그러면 L1 은 "이론적 exact"가 아니라 물리 기반 결정적 통계량 + 보정 임계가 됩니다 —
정직하게 그렇게 표기합니다. (L2 의 CP 와 같은 원리이며, 둘의 차이는 통계량의 출처입니다.)

In [9]:
# ============================================================
# [B8b] L1 진단 + 임계 재calibration
# ============================================================
from scipy.stats import chi2
from safe_logic import SafeLogic, l1_chi2_flag, l2_cp_flag

# --- 진단 1: 잔차의 백색성 (lag-1 자기상관) ---
acs = []
for fid in TRAIN_IDS[:60]:
    zf = flat.loc[flat['flight_id'] == fid, 'z'].to_numpy()
    if len(zf) > 10:
        acs.append(float(np.corrcoef(zf[:-1], zf[1:])[0, 1]))
AC1 = float(np.mean(acs))
print(f"잔차 lag-1 자기상관 (train 60비행) = {AC1:+.4f}")
print("  → 0 이어야 백색. 이 값이 유의하면 창 내 chi2 합이 chi2(W) 를 따르지 않는다.")

# --- 진단 2: 이론 임계의 실제 FPR ---
q_chi_theory = float(chi2.ppf(1 - ALPHA, W))
fpr_theory_win = float((CHIw[M_TEST] > q_chi_theory).mean() * 100)
print(f"\n이론 임계 chi2({W}) {100*(1-ALPHA):.0f}% = {q_chi_theory:.3f}")
print(f"  정상 test 창 FPR = {fpr_theory_win:.1f}%   (목표 {ALPHA*100:.0f}%)")
print("  → 이론 보장이 성립하지 않는다.")

# --- 오진 기록: 분산 스케일 보정은 역효과 ---
RHO_BAD = float(np.mean(Xw[M_TRAIN, 1] ** 2))
fpr_rho = float((CHIw[M_TEST] / RHO_BAD > q_chi_theory).mean() * 100)
print(f"\n[오진 기록] rho=E[z_std^2]={RHO_BAD:.4f} 로 S 를 보정하면 오히려")
print(f"  FPR 이 {fpr_theory_win:.1f}% → {fpr_rho:.1f}% 로 악화된다. 창 내 표준편차는")
print("  저주파 드리프트를 과소평가하므로 분산 추정치가 아니다.")

# --- 재calibration: 임계를 calibration 창에서 경험적으로 ---
q_chi = cp_threshold(CHIw[M_CAL], ALPHA)
print(f"\n재calibration: q_chi = {q_chi:.3f}  (calibration 창 {int(M_CAL.sum())}개)")
print(f"{'구간':<14}{'창 수':>8}{'초과율':>10}")
for tag, msk in (('train', M_TRAIN), ('cal', M_CAL), ('test(정상)', M_TEST), ('OOD', M_OOD)):
    print(f"{tag:<14}{int(msk.sum()):>8}{float((CHIw[msk] > q_chi).mean()*100):>9.1f}%")
print(f"\n  → 정상 test 창 FPR 이 목표 {ALPHA*100:.0f}% 안으로 들어온다")
print(f"     ({fpr_theory_win:.1f}% → {float((CHIw[M_TEST] > q_chi).mean()*100):.1f}%)")
print("  ※ 잔차 백색화(AR 등)를 하면 이론 임계를 되살릴 수 있다 — 범위 밖, 남은 과제.")

잔차 lag-1 자기상관 (train 60비행) = +0.3994
  → 0 이어야 백색. 이 값이 유의하면 창 내 chi2 합이 chi2(W) 를 따르지 않는다.

이론 임계 chi2(25) 90% = 34.382
  정상 test 창 FPR = 34.4%   (목표 10%)
  → 이론 보장이 성립하지 않는다.

[오진 기록] rho=E[z_std^2]=0.5503 로 S 를 보정하면 오히려
  FPR 이 34.4% → 56.3% 로 악화된다. 창 내 표준편차는
  저주파 드리프트를 과소평가하므로 분산 추정치가 아니다.

재calibration: q_chi = 84.210  (calibration 창 1579개)
구간                 창 수       초과율
train             3162     13.0%
cal               1579      9.9%
test(정상)          1610     11.7%
OOD               1369     40.6%

  → 정상 test 창 FPR 이 목표 10% 안으로 들어온다
     (34.4% → 11.7%)
  ※ 잔차 백색화(AR 등)를 하면 이론 임계를 되살릴 수 있다 — 범위 밖, 남은 과제.


In [10]:
# ============================================================
# [B9] 비교 매트릭스 — 누가 L1 을 이기는가
# ============================================================
chi_ok = lambda idx: CHIw[idx] > q_chi     # B8b 에서 재calibration 한 임계
def ev(name, flag_mask):
    # flag_mask: 창 단위 불리언. 정상 test FPR 과 OOD TPR 을 반환한다.
    m_test = M_TEST & flag_mask
    m_ood = M_OOD & flag_mask
    fpr = m_test.sum() / max(M_TEST.sum(), 1) * 100
    tpr = m_ood.sum() / max(M_OOD.sum(), 1) * 100
    fam = {}
    for t in np.unique(OOD_TYPE[M_OOD]):
        sel = M_OOD & (OOD_TYPE == t)
        fam[t] = flag_mask[sel].mean() * 100
    return fpr, tpr, fam

results = {}
# L0 는 센서 하드 바운드 — 이 데이터에서는 발동하지 않는다
results['L1 chi2 (이론 임계)'] = ev('chi2', chi_ok(np.arange(len(CHIw))))

def score_all(fn):
    # test(정상) + OOD 창에만 스코어를 계산한다 (train/cal 은 누수 방지로 제외)
    out = np.full(len(CHIw), np.nan)
    out[M_TEST | M_OOD] = fn(S(M_TEST | M_OOD))
    return out
s_svm, s_kde, s_kpca = score_all(score_svm), score_all(score_kde), score_all(score_kpca)
results['OC-SVM + CP'] = ev('svm', np.where(np.isnan(s_svm), False, s_svm > q_svm))
results['KDE + CP'] = ev('kde', np.where(np.isnan(s_kde), False, s_kde > q_kde))
results['kPCA + CP'] = ev('kpca', np.where(np.isnan(s_kpca), False, s_kpca > q_kpca))
# 단일 통계량 + 경험적 95분위 (참고 기준선)
zmed = Xw[:, 1]  # 창별 z_std
q_emp = float(np.percentile(zmed[M_CAL], 95))
results['참고: 경험적 95분위'] = ev('emp', np.where(np.isnan(s_svm), False, zmed > q_emp))

fams = sorted({t for _, _, f in results.values() for t in f})
hdr = f"{'방법':<22}{'FPR(정상)':>10}{'TPR(OOD)':>10}" + "".join(f"{f:>8}" for f in fams)
print("=" * len(hdr))
print("비교 매트릭스  (FPR 목표 ≤ 10%)")
print("=" * len(hdr))
print(hdr)
print("-" * len(hdr))
for name, (fpr, tpr, fam_d) in results.items():
    print(f"{name:<22}{fpr:>9.1f}%{tpr:>9.1f}%" + "".join(f"{fam_d.get(f, 0):>7.0f}%" for f in fams))
print("=" * len(hdr))
print(f"\nOOD 창 {int(M_OOD.sum())}개 / 정상 test 창 {int(M_TEST.sum())}개")
print("⚠️ 위 FPR 이 α=10% 를 넘는다 — 이유는 아래 B9b 에서 규명한다.")

비교 매트릭스  (FPR 목표 ≤ 10%)
방법                       FPR(정상)  TPR(OOD)    drag    loft    mass    wind
--------------------------------------------------------------------------
L1 chi2 (이론 임계)            11.7%     40.6%     50%     21%     67%     17%
OC-SVM + CP                12.2%     33.1%     40%     13%     65%      8%
KDE + CP                   11.6%     34.0%     41%     15%     65%      9%
kPCA + CP                  12.0%     33.7%     42%     13%     65%      8%
참고: 경험적 95분위                3.7%      6.4%      4%      2%     16%      3%

OOD 창 1369개 / 정상 test 창 1610개
⚠️ 위 FPR 이 α=10% 를 넘는다 — 이유는 아래 B9b 에서 규명한다.


## B9b. 비행 단위 CP — 계획서가 요구하는 보장

### 왜 창 단위 CP 가 α 를 못 지키는가

CP 의 유한표본 보장은 점들이 교환가능할 때 성립합니다. 그런데 같은 비행의 창들은
강하게 상관되어 있어, 창 1,579개는 사실상 비행 50편의 정보량밖에 갖지 않습니다.
실효 표본수가 50이면 분위수 추정이 거칠어져 FPR 이 α 를 넘습니다.

### 운영 판단은 비행 단위다

"이 비행을 중단시킬 것인가"는 비행당 한 번의 결정입니다. 그러므로
비행별로 창 스코어를 집계한 뒤 비행 단위로 conformal calibration 해야 합니다.
(계획서의 "비행 단위 FPR" — conformal-safety-monitoring 의 연구 novelty 1위)

아래 셀은 비행별 집계 스코어(창 스코어의 95분위)에 대해 CP 를 다시 걸고,
창 단위 결과와 나란히 비교합니다.

In [11]:
# ============================================================
# [B9b] 비행 단위 CP
# ============================================================
# 집계 스코어를 cal/test/OOD 비행 전체에 대해 계산한다
M_CTO = M_CAL | M_TEST | M_OOD
def score_union(fn):
    out = np.full(len(CHIw), np.nan)
    out[M_CTO] = fn(S(M_CTO))
    return out

def flight_agg(score_arr, ids, q=95):
    # 비행별 집계 = 그 비행 창 스코어의 q 분위 (최악 구간을 반영)
    vals = []
    for fid in ids:
        s = score_arr[IDw == fid]
        s = s[~np.isnan(s)]
        vals.append(float(np.percentile(s, q)) if len(s) else np.nan)
    return np.array(vals)

def ood_ids_of(t):
    return fl.loc[(fl.ood == 1) & (fl['type'] == t), 'fid'].to_numpy()

def flight_level(fn, alpha=ALPHA, q=95):
    sa = score_union(fn)
    thr = cp_threshold(flight_agg(sa, CAL_IDS, q), alpha)   # 비행 단위 calibration
    fpr = float((flight_agg(sa, TEST_IDS, q) > thr).mean() * 100)
    oo = flight_agg(sa, ood_ids, q)
    tpr = float((oo > thr).mean() * 100)
    fam = {t: float((flight_agg(sa, ood_ids_of(t), q) > thr).mean() * 100)
           for t in np.unique(OOD_TYPE[M_OOD])}
    return thr, fpr, tpr, fam

# chi2 는 보정된 이론 임계 + N-of-M 비행 규칙으로 평가한다 (결정 D 의 방식)
def flight_nofm(flag_win, ids, n=3, m=5):
    # 최근 m 창 중 n 이상 플래그면 그 비행은 abort (해제는 5연속 정상)
    res = {}
    for fid in ids:
        idx = np.where(IDw == fid)[0]
        sl = SafeLogic(n=n, m=m, hysteresis_clear=5)
        st = [sl.decide(l0=False, l1=bool(flag_win[i]), l2=False).state for i in idx]
        res[int(fid)] = (st[-1] == 'aborted')
    return res

_l1 = flight_nofm(CHIw > q_chi, np.concatenate([TEST_IDS, ood_ids]))
_l1_fpr = float(np.mean([_l1[int(f)] for f in TEST_IDS]) * 100)
_l1_tpr = float(np.mean([_l1[int(f)] for f in ood_ids]) * 100)
_l1_fam = {t: float(np.mean([_l1[int(f)] for f in ood_ids_of(t)]) * 100)
           for t in np.unique(OOD_TYPE[M_OOD])}

rows = [('L1 chi2 (보정임계+N-of-M)', (q_chi, _l1_fpr, _l1_tpr, _l1_fam)),
        ('OC-SVM + CP', flight_level(score_svm)),
        ('KDE + CP', flight_level(score_kde)), ('kPCA + CP', flight_level(score_kpca))]
fams = sorted({t for _, res in rows for t in res[3]})
hdr = f"{'방법':<22}{'임계':>9}{'FPR(비행)':>11}{'TPR(비행)':>11}" + "".join(f"{t:>8}" for t in fams)
print("=" * len(hdr))
print("비행 단위 비교 매트릭스  (FPR 목표 ≤ 10%, calibration 50비행)")
print("=" * len(hdr))
print(hdr); print("-" * len(hdr))
for name, (thr, fpr, tpr, fam) in rows:
    print(f"{name:<22}{thr:>9.3f}{fpr:>10.1f}%{tpr:>10.1f}%" + "".join(f"{fam.get(t,0):>7.0f}%" for t in fams))
print("=" * len(hdr))
print(f"\nOOD 비행 {len(ood_ids)}편 / 정상 test 비행 {len(TEST_IDS)}편")

비행 단위 비교 매트릭스  (FPR 목표 ≤ 10%, calibration 50비행)
방법                           임계    FPR(비행)    TPR(비행)    drag    loft    mass    wind
-------------------------------------------------------------------------------------
L1 chi2 (보정임계+N-of-M)    84.210       6.0%      68.0%    100%     56%     93%      0%
OC-SVM + CP              13.002       6.0%      54.0%     67%     11%    100%      9%
KDE + CP                  9.714       6.0%      60.0%     80%     22%    100%      9%
kPCA + CP                 3.342       4.0%      54.0%     67%     11%    100%      9%

OOD 비행 50편 / 정상 test 비행 50편


In [12]:
# ============================================================
# [B10] 단계별(phase-conditional) calibration
# ============================================================
# 비행은 시계열이고 연소/관성 구간의 잔차 분포가 다르다 → marginal 보장이 깨진다.
# 위상별로 따로 calibration 하면 위상별 FPR 보장이 회복된다.
BURN = PHw[M_CAL] > 0.5
q_by_phase = {}
for tag, mask in (('연소중', True), ('관성', False)):
    cal_sel = M_CAL & ((PHw > 0.5) == mask)
    if cal_sel.sum() < 10:
        print(f"  {tag}: calibration 창 부족({int(cal_sel.sum())}) → 건너뜀"); continue
    q_by_phase[tag] = cp_threshold(score_svm(S(cal_sel)), ALPHA)
print("위상별 CP 임계값 (OC-SVM):", {k: round(v, 4) for k, v in q_by_phase.items()})
print(f"(참고: marginal 임계 q = {q_svm:.4f})\n")

def ev_phase(tag, mask):
    sel_t = M_TEST & (PHw > 0.5) == mask
    sel_o = M_OOD & (PHw > 0.5) == mask
    q = q_by_phase.get(tag, q_svm)
    f = np.full(len(CHIw), np.nan); f[M_TEST | M_OOD] = score_svm(S(M_TEST | M_OOD))
    fl_t = np.where(np.isnan(f), False, f > q) & sel_t
    fl_o = np.where(np.isnan(f), False, f > q) & sel_o
    return (fl_t.sum() / max(sel_t.sum(), 1) * 100, fl_o.sum() / max(sel_o.sum(), 1) * 100)

print(f"{'위상':<10}{'정상창':>8}{'OOD창':>8}{'FPR':>8}{'TPR':>8}")
for tag, mask in (('연소중', True), ('관성', False)):
    nt = int((M_TEST & ((PHw > 0.5) == mask)).sum()); no = int((M_OOD & ((PHw > 0.5) == mask)).sum())
    fpr, tpr = ev_phase(tag, mask)
    print(f"{tag:<10}{nt:>8}{no:>8}{fpr:>7.1f}%{tpr:>7.1f}%")
print("\n→ marginal 임계 하나로는 위상별 FPR 이 α 에서 벗어난다. 위상별 calibration 이 이를 복구한다.")

위상별 CP 임계값 (OC-SVM): {'연소중': 0.3516, '관성': -0.3575}
(참고: marginal 임계 q = 0.1251)

위상             정상창    OOD창     FPR     TPR
연소중            700     700   12.9%   22.3%


관성             910     669    8.5%    7.8%

→ marginal 임계 하나로는 위상별 FPR 이 α 에서 벗어난다. 위상별 calibration 이 이를 복구한다.


In [13]:
# ============================================================
# [B11] Safe-Logic 2.0 연동 — N-of-M + 히스테리시스
# ============================================================
# L1 은 B8b 에서 보정한 통계량 + 이론 임계 (Mehra 정합) 를 쓴다.
# L2 임계는 **비행 단위** CP 임계를 쓴다 (창 단위 임계는 FPR 을 못 지킨다 — B9/B9b)
Q_L2, FPR_L2, TPR_L2, _ = flight_level(score_svm)
print("safe_logic.SafeLogic(n=3, m=5, hysteresis_clear=5) — 결정 D 와 일치")
print(f"L1: chi2 보정 임계 {q_chi:.3f}  (이론값 {q_chi_theory:.3f} → 백색성 위반으로 재calibration)")
print(f"L2: 비행 단위 CP 임계 {Q_L2:.4f}   [비행 FPR {FPR_L2:.1f}% / TPR {TPR_L2:.1f}%]\n")

SA_SVM = score_union(score_svm)      # 비행 단위 집계용 스코어 (창 단위 값)

def flight_trace(fid, use_l1=False, use_l2=True):
    idx = np.where(IDw == fid)[0]
    sl = SafeLogic(n=3, m=5, hysteresis_clear=5)
    dec = []
    for i in idx:
        # l1_chi2_flag 는 내부에서 이론 임계(chi2.ppf)를 쓰므로 백색성 위반 데이터에는
        # 부적합하다. B8b 에서 재calibration 한 임계를 직접 적용한다.
        l1 = bool(CHIw[i] > q_chi) if use_l1 else False
        s = float(SA_SVM[i])
        l2 = l2_cp_flag(s, Q_L2) if (use_l2 and not np.isnan(s)) else False
        dec.append(sl.decide(l0=False, l1=l1, l2=l2).state)
    return np.array(dec), sl

print(f"{'비행':<18}{'채널':<10}{'abort 창':>10}{'최종':>10}{'L2투표':>8}{'정상연속':>10}")
for fid, tag in ((int(TEST_IDS[0]), '정상'), (int(ood_ids[0]), 'OOD')):
    for use_l1, label in ((False, 'L2만'), (True, 'L1+L2')):
        d, sl = flight_trace(fid, use_l1=use_l1)
        verdict = 'ABORT' if d[-1] == 'aborted' else 'NORMAL'
        print(f"fid {fid:<14}({tag:4s}) {label:<10}{int((d=='aborted').sum()):>8}/{len(d):<3}"
              f"{verdict:>10}{sl.l2_votes:>6}/{sl.m:<3}{sl.normal_streak:>10}")

print()
print("→ L1·L2 각 채널이 자기 임계를 지키며 계층을 이룬다 (결정 D).")
print("   비행 단위 abort 비율을 집계하면:")
print(f"{'채널':<10}{'정상 abort':>13}{'OOD abort':>12}")
for _l1, _l2, _tag in ((False, True, 'L2만'), (True, False, 'L1만'), (True, True, 'L1+L2')):
    _t = 0; _o = 0
    for _f in TEST_IDS:
        _t += (flight_trace(int(_f), use_l1=_l1, use_l2=_l2)[0][-1] == 'aborted')
    for _f in ood_ids:
        _o += (flight_trace(int(_f), use_l1=_l1, use_l2=_l2)[0][-1] == 'aborted')
    print(f"{_tag:<10}{_t/len(TEST_IDS)*100:>12.1f}%{_o/len(ood_ids)*100:>11.1f}%")
print()
print("   ※ 위 데모에서 정상 비행이 L1 에 걸린 것은 정상이다 — 비행 단위 FPR 이 0 이 아니므로")
print("     (기대 6%) 50편 중 몇 편은 걸린다. 문제는 '체계적 오탐'이 없어야 한다는 것이고,")
print("     정상 abort 율이 설계 목표 alpha 이하이므로 그 조건은 만족한다.")
print("→ 남은 과제: N-of-M 규칙 전체를 하나의 conformal 객체로 calibration (비행 단위 FPR).")

safe_logic.SafeLogic(n=3, m=5, hysteresis_clear=5) — 결정 D 와 일치
L1: chi2 보정 임계 84.210  (이론값 34.382 → 백색성 위반으로 재calibration)
L2: 비행 단위 CP 임계 13.0016   [비행 FPR 6.0% / TPR 54.0%]



비행                채널           abort 창        최종    L2투표      정상연속
fid 51            (정상  ) L2만              0/30     NORMAL     0/5           0
fid 51            (정상  ) L1+L2           16/30      ABORT     0/5           0
fid 1000          (OOD ) L2만              5/28      ABORT     3/5           4
fid 1000          (OOD ) L1+L2           12/28      ABORT     0/5           0

→ L1·L2 각 채널이 자기 임계를 지키며 계층을 이룬다 (결정 D).
   비행 단위 abort 비율을 집계하면:
채널             정상 abort   OOD abort
L2만                0.0%       32.0%
L1만                6.0%       68.0%
L1+L2              6.0%       68.0%

   ※ 위 데모에서 정상 비행이 L1 에 걸린 것은 정상이다 — 비행 단위 FPR 이 0 이 아니므로
     (기대 6%) 50편 중 몇 편은 걸린다. 문제는 '체계적 오탐'이 없어야 한다는 것이고,
     정상 abort 율이 설계 목표 alpha 이하이므로 그 조건은 만족한다.
→ 남은 과제: N-of-M 규칙 전체를 하나의 conformal 객체로 calibration (비행 단위 FPR).


## B12. (선택) MC Dropout 기준선 비교 — torch 필요

Step 5 의 MC Dropout 은 예측 불확실성으로 OOD 를 탐지합니다. 이 노트북의 커널+CP 와
같은 FPR 에서 TPR 을 비교하는 것이 계획서 §4.3-Q1 입니다.

단, 이 비교는 GPU/CPU torch 와 Step 4 모델(`best_gru_model.pth`)이 필요합니다.
이 저장소의 기본 환경(`requirements.txt`)에는 torch 가 포함되지만, 이 머신에서의 검증은
미완이므로 아래 셀은 torch 가 있을 때만 실행되고 없으면 건너뜁니다.

> 커널+CP 의 장점은 물리 잔차만 쓰므로 학습된 예측모델에 의존하지 않는다는 것입니다
> (관심사 분리). MC Dropout 은 예측이 틀리면 OOD 탐지도 같이 틀어집니다.

In [14]:
# ============================================================
# [B12] (선택) MC Dropout 비교 — torch 없으면 건너뜀
# ============================================================
import importlib.util
if importlib.util.find_spec('torch') is None:
    print("torch 미설치 → MC Dropout 비교를 건너뜁니다.")
    print("  설치: pip install torch   (그 뒤 이 셀을 다시 실행)")
    print("  필요한 파일: ./data/simulated/best_gru_model.pth, input_scaler.pkl, model_metadata.pkl")
else:
    import torch
    print(f"torch {torch.__version__} 발견 — Step 5 MC Dropout 기준선을 계산하세요.")
    print("  (Step 5 노트북의 predict_with_uncertainty 를 재사용하는 것이 다음 작업입니다)")

torch 미설치 → MC Dropout 비교를 건너뜁니다.
  설치: pip install torch   (그 뒤 이 셀을 다시 실행)
  필요한 파일: ./data/simulated/best_gru_model.pth, input_scaler.pkl, model_metadata.pkl


## 요약

| 항목 | 내용 |
|:--|:--|
| 입력 | 명목 물리모델 잔차 (`scripts/model_filter.py` → `residuals.npz`) |
| 잔차 모델 | 명목 4.0 kg / Cd 1.0 / 80도, 레일 2 m, ISA 대기 |
| 분할 | 비행 단위 (정상 100 train / 50 cal / 50 test + OOD 50 test) |
| L1 | χ² 검정 — 보정 임계 (이론값은 백색성 위반으로 성립하지 않음, B8b) |
| L2 | OC-SVM(RBF) + CP, KDE·kPCA 는 비교 |
| 보장 | CP: 교환가능성 하에서 P(오탐) ≤ α 유한표본 |
| 결정 | Safe-Logic 2.0 — L0/L1 결정적, L2 N-of-M + 히스테리시스 |

### 정직한 한계

1. 시뮬 데이터 — CP 보장은 calibration 분포 안에서만 유효. 실측과의 sim-to-real 이동이 있다.
2. OOD 정의가 협소 — 파라미터 범위 이탈(mass/Cd/wind/launch_angle)로 제한. 실세계 고장 모드와 다르다.
3. 위상이 2개뿐 — 데이터가 아포지에서 잘려 부스트/관성만 존재. 계획서의 4위상이 아니다.
4. L1 의 이론적 exact 보장이 성립하지 않는다 — 잔차가 백색이 아니기 때문(자기상관, B8b 에서
   실측). 임계를 calibration 창에서 경험적으로 재설정해 FPR 을 맞췄다. 잔차 백색화(AR 등)를
   하면 이론 임계를 되살릴 수 있다 — 남은 과제.
5. detection 이 계열 선택적 — 질량 계열은 잘 잡히지만 강풍 계열은 거의 안 잡힌다.
6. CP 창↔비행 단위 차이 — 창 단위는 FPR 이 α 를 초과. 같은 비행의 창이 상관되어 실효
   표본수가 비행 수에 불과하기 때문. 비행 단위로 calibration 해야 보장이 선다 (B9b).
7. MC Dropout 비교 미완 — torch 미설치.

### 재현

```bash
python scripts/model_filter.py          # 잔차 생성
python scripts/generate_ood.py          # OOD 세트 생성
jupyter lab "step5.5-kernel-anomaly-detection.ipynb"
```